In [26]:
from df import enhance, init_df

from torch_stoi import NegSTOILoss
from torchmetrics.audio.pesq import PerceptualEvaluationSpeechQuality
from torchmetrics.audio import SpeechReverberationModulationEnergyRatio, ShortTimeObjectiveIntelligibility, DeepNoiseSuppressionMeanOpinionScore
from torchaudio.transforms import Resample

import torch
import torchaudio
import numpy as np

import os

In [27]:
SEED = 1984

np.random.seed(SEED)
torch.manual_seed(SEED)

gen = torch.Generator()
gen.manual_seed(SEED)

np.set_printoptions(precision=3)
torch.set_printoptions(precision=3)

In [28]:
import yaml

from NISQA_s.src.core.model_torch import model_init
from NISQA_s.src.utils.process_utils import process

NISQA_PATH = "NISQA_s/config/nisqa_s.yaml"

with open(NISQA_PATH, 'r') as stream:
    nisqa_args = yaml.safe_load(stream)
nisqa_args["ms_n_fft"] = 512
nisqa_args["hop_length"] = 256
nisqa_args["ms_win_length"] = 512
nisqa_args["ckp"] = nisqa_args["ckp"][3:]

nisqa, h0_nisqa, c0_nisqa = model_init(nisqa_args)

/home/zakhar/miniconda3/envs/ems_dereverb/lib/python3.10/site-packages/torch/nn/modules/rnn.py:83: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=1 and num_layers=1
  warnings.warn("dropout option adds dropout after all but last "


In [29]:
SR = 48_000

# NOISE_PATH = "data/DS_10283_2791/clean_testset_wav"
CLEAN_PATH = "data/DS_10283_2791/clean_testset_wav"
NOISE_PATH = "data/demand_test"
RIRS = {1: os.path.join("data", "rirs48_large_3_test"), 1: os.path.join("data", "rirs48_super_large_3_test")}
# noise_paths = [os.path.join(NOISE_PATH, x) for x in os.listdir(NOISE_PATH)]
# clean_paths = [os.path.join(CLEAN_PATH, x) for x in os.listdir(CLEAN_PATH)]

# test_data = list(zip(noise_paths, clean_paths))

BATCH_SIZE = 32
DEVICE = "cuda:0"

In [30]:
from src.dataset import *

test_dataset = TRUNetDataset(CLEAN_PATH, sr=SR, noise_dir=NOISE_PATH, rir_dir=RIRS, snr=[0, 5, 10, 15], rir_proba=0.85, noise_proba=0.85, rir_target=False, return_noise=False, return_rir=False)
test_dataset.set_epoch(1)

36
12


In [31]:
tmp1, tmp2, _, _ = test_dataset[4]

In [32]:
from IPython.display import Audio
Audio(tmp1, rate=SR)

In [33]:
srmr = SpeechReverberationModulationEnergyRatio(fs=16_000, norm=False)
stoi = NegSTOILoss(SR, use_vad=False, do_resample=False).to(DEVICE)
pesq = PerceptualEvaluationSpeechQuality(fs=16_000, mode="wb").to(DEVICE)
dnsmos = DeepNoiseSuppressionMeanOpinionScore(16_000, False, device=DEVICE)

In [34]:
from df import enhance, init_df

model, df_state, _ = init_df()

2026-04-20 16:42:48 | INFO     | DF | Loading model settings of DeepFilterNet3


2026-04-20 16:42:48 | INFO     | DF | Using DeepFilterNet3 model at /home/zakhar/.cache/DeepFilterNet/DeepFilterNet3
2026-04-20 16:42:48 | INFO     | DF | Initializing model `deepfilternet3`
2026-04-20 16:42:48 | INFO     | DF | Found checkpoint /home/zakhar/.cache/DeepFilterNet/DeepFilterNet3/checkpoints/model_120.ckpt.best with epoch 120
2026-04-20 16:42:48 | INFO     | DF | Running on device cuda:0
2026-04-20 16:42:48 | INFO     | DF | Model loaded


In [35]:
input_signal, target_signal, _, _ = test_dataset[3]
Audio(input_signal.detach().numpy(), rate=SR)

In [40]:
from tqdm import tqdm
from scipy.io.wavfile import write

def get_metrics(data, device="cpu"):
    nisqa_scores = []
    pesq_scores = []
    stoi_scores = []
    srmr_scores = []
    dns_scores = []
    with torch.no_grad():
        for ind, (input_signal, target_signal, _, _) in tqdm(enumerate(data)):
            
            # signal, signal_sr = torchaudio.load(input_path)
            # target, target_sr = torchaudio.load(target_path)

            input_signal = input_signal.to(device)
            target_signal = target_signal.to(device)
            
            # print(signal.shape)
            output = enhance(model, df_state, input_signal.cpu()).to(device) # torch.from_numpy(enhancer(signal[0].cpu(), signal_sr)).unsqueeze(0).to(device)

            write(f'dfn_in_hard/intput_{ind}.wav', SR, input_signal.cpu().detach().numpy()[0])
            write(f'dfn_out_hard/output_{ind}.wav', SR, output.cpu().detach().numpy()[0])

            min_l = min(output.shape[-1], target_signal.shape[-1])

            nisqa_score, _, _ = process(output.detach().cpu(), SR, nisqa, h0_nisqa, c0_nisqa, nisqa_args)

            stoi_score = stoi(output[..., :min_l], target_signal[..., :min_l])
            srmr_score = srmr(output.detach().cpu())
            
            resampler = Resample(SR, 16_000)
            output = resampler(output.cpu()).cuda()
            target_signal = resampler(target_signal.cpu()).cuda()
            
            min_l = min(output.shape[-1], target_signal.shape[-1])

            srmr_score = srmr(output.detach().cpu())
            dnsmos_score = dnsmos(output.detach())

            try:
                pesq_score = pesq(output[..., :min_l], target_signal[..., :min_l])
            except Exception as e:
                # print(min_l)
                # out_wave_ = output.reshape(-1)
                # target_ = target.reshape(-1)
                # write('exception_out.wav', SR, out_wave_.cpu().detach().numpy())
                # write('exception_in.wav', SR, target_.cpu().detach().numpy())
                continue

            nisqa_scores.append(nisqa_score[0])
            srmr_scores.append(srmr_score)
            stoi_scores.append(stoi_score.cpu())
            pesq_scores.append(pesq_score.cpu())
            dns_scores.append(dnsmos_score.cpu())

    nisqa_scores = torch.vstack(nisqa_scores).mean(dim=0)
    stoi_scores = torch.vstack(stoi_scores).mean(dim=0)
    srmr_scores = torch.vstack(srmr_scores).mean(dim=0)
    pesq_scores = torch.vstack(pesq_scores).mean(dim=0)
    dns_scores = torch.vstack(dns_scores).mean(dim=0)

    result = {"nisqa": nisqa_scores, "stoi": stoi_scores, "srmr": srmr_scores, "pesq": pesq_scores, "dnsmos": dns_scores}
        
    return result

In [41]:
metrics = get_metrics(test_dataset, device=DEVICE)

824it [20:53,  1.52s/it]


In [ ]:
print("NISQA score (MOS, NOI, DISC, COL, LOUD):", metrics['nisqa'])
print(f"STOI score: {-metrics['stoi']}")
print(f"SRMR score: {metrics['srmr']}")
print(f"PESQ-WB score: {metrics['pesq']}")
print(f"DNSMOS score: {metrics['dnsmos']}")

NISQA score (MOS, NOI, DISC, COL, LOUD): tensor([4.133, 4.292, 4.041, 4.064, 4.139])
STOI score: tensor([0.905])
SRMR score: tensor([9.269])
PESQ-WB score: tensor([2.723])
DNSMOS score: tensor([3.549, 3.360, 4.039, 3.095], dtype=torch.float64)
